# Lab 10: Regression Diagnostics and Extensions
> Week 10 | CLO3 | ISLP Ch.3.3

## บทนำสัปดาห์

สัปดาห์นี้เราจะขยาย regression ด้วยสามส่วนสำคัญ: (1) **Qualitative predictors** ผ่าน dummy variables ที่ช่วยรวม categorical data เข้า regression ได้, (2) **Extensions** ได้แก่ interaction terms ที่จับ synergy ระหว่าง predictors และ polynomial regression สำหรับความสัมพันธ์ที่ไม่เป็นเส้นตรง, (3) **Diagnostics** ซึ่งเป็นขั้นตอนที่ Data Scientist ต้องทำทุกครั้งก่อนเชื่อผล regression เพราะ R² สูงและ p-value น้อยไม่ได้รับประกันว่า model ถูกต้อง การตรวจ residual plots, VIF และ Cook's Distance ช่วยให้มั่นใจว่า inference ที่ได้น่าเชื่อถือ ทักษะเหล่านี้เป็น mandatory ก่อน publish หรือเสนอผลต่อผู้บริหาร

**LLo**: รวม qualitative predictor เข้า regression, ตรวจสอบ potential problems ด้วย diagnostic plots และแก้ปัญหาได้

**สิ่งที่จะเรียนรู้**:
- Part 1: Dummy variables — categorical predictor ใน regression
- Part 2: Interaction terms + polynomial regression
- Part 3: Residual diagnostics — 4 standard plots
- Part 4: VIF — multicollinearity detection
- Part 5: Case Study — full diagnostics workflow


In [ ]:
# ─── ติดตั้ง ISLP package ───────────────────────────────────────────
# วัตถุประสงค์: ติดตั้งไลบรารี ISLP ซึ่งเป็นแหล่งข้อมูลทางการของหนังสือ
# (James, Witten, Hastie, Tibshirani, Taylor) ที่รวม dataset ส่วนใหญ่ในเล่มไว้ให้โหลดตรง ๆ
# อ้างอิง: statlearning.com -> Python package "ISLP"
!pip install -q ISLP
print('ติดตั้ง ISLP เสร็จแล้ว')

In [ ]:
# ─── Import libraries ──────────────────────────────────────────────
# วัตถุประสงค์: โหลด library ทั้งหมดที่ใช้ใน lab
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor, OLSInfluence
from statsmodels.graphics.gofplots import ProbPlot
from scipy import stats

plt.rcParams['figure.figsize'] = (10, 6)
np.random.seed(42)
print('Libraries loaded ✓')

In [ ]:
# ─── โหลด datasets ─────────────────────────────────────────────────
# วัตถุประสงค์: โหลด Advertising และ Auto dataset สำหรับ lab
# ลำดับการหาแหล่งข้อมูล: 1) ISLP package  2) URL/ไฟล์ท้องถิ่น  3) synthetic data จำลอง
#
# หมายเหตุ: Auto อยู่ใน ISLP package จริง (load_data('Auto') ใช้ได้ตรง ๆ)
# ส่วน Advertising ไม่ได้ฝังอยู่ในแพ็กเกจ (เป็นตัวอย่างเปิดเรื่องใน Ch.2 ไม่ใช่ Lab
# dataset ใน Table 1.1) จึงลองก่อนแล้วปล่อยให้ตกไปใช้ URL ตามที่หนังสือสอนจริง
try:
    from ISLP import load_data
    df_adv = load_data('Advertising')
    print('Advertising: loaded via ISLP package ✓')
except Exception:
    try:
        df_adv = pd.read_csv('https://www.statlearning.com/s/Advertising.csv', index_col=0)
        print('Advertising: loaded via statlearning.com URL ✓')
    except Exception:
        np.random.seed(0)
        n = 200
        TV    = np.random.uniform(0.7, 296.4, n)
        Radio = np.random.uniform(0, 49.6, n)
        News  = np.random.uniform(0.3, 114, n)
        Sales = 2.94 + 0.046*TV + 0.189*Radio - 0.001*News + np.random.normal(0,1.69,n)
        df_adv = pd.DataFrame({'TV':TV,'Radio':Radio,'Newspaper':News,'Sales':Sales})
        print('Advertising: using synthetic data')

# Auto dataset — อยู่ใน ISLP package โดยตรง
try:
    from ISLP import load_data
    df_auto = load_data('Auto')
    print('Auto: loaded via ISLP package ✓')
except Exception:
    try:
        df_auto = pd.read_csv('Auto.csv', na_values='?').dropna()
        print('Auto: loaded via local Auto.csv ✓')
    except Exception:
        np.random.seed(1)
        n = 392
        hp = np.random.uniform(46, 230, n)
        mpg = 50 - 0.15*hp + 0.0003*hp**2 + np.random.normal(0, 3, n)
        df_auto = pd.DataFrame({'horsepower': hp, 'mpg': mpg})
        print('Auto: using synthetic data')

print(f'Advertising shape: {df_adv.shape}')
print(f'Auto shape: {df_auto.shape}')

---
## Part 1: Dummy Variables — Categorical Predictor ใน Regression

**Part นี้เราจะเพิ่ม categorical predictor เข้า regression** เพื่อให้เห็นว่า dummy variable ทำให้ regression มี intercept ต่างกันตามกลุ่ม

ก่อนอื่นต้องเข้าใจว่า regression ต้องการตัวเลข แต่ categorical variable เช่น "Yes"/"No" ไม่ใช่ตัวเลข — dummy encoding แก้ปัญหานี้โดยสร้าง 0/1 indicator variable


In [ ]:
# ─── สร้าง categorical TV_level ────────────────────────────────────
# วัตถุประสงค์: แบ่ง TV budget เป็น Low/High เพื่อใช้ใน regression
df_adv['TV_level'] = pd.cut(df_adv['TV'],
                            bins=[0, 100, 300],
                            labels=['Low', 'High'])

print('TV_level distribution:')
print(df_adv['TV_level'].value_counts())

# ─── Fit: Sales ~ TV_level ─────────────────────────────────────────
# วัตถุประสงค์: ดูว่า High-TV market มี Sales สูงกว่า Low-TV เฉลี่ยเท่าไร
model_dummy = smf.ols('Sales ~ C(TV_level)', data=df_adv).fit()
print('\n', model_dummy.summary())

In [ ]:
# ─── Visualize: parallel regression lines ──────────────────────────
# วัตถุประสงค์: เห็นว่า dummy predictor สร้าง intercept ต่างกัน แต่ slope เหมือนกัน
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Boxplot: Sales by TV_level
df_adv.boxplot(column='Sales', by='TV_level', ax=axes[0])
axes[0].set_title('Sales by TV Level')
axes[0].set_xlabel('TV Level'); axes[0].set_ylabel('Sales')

# Scatter: TV vs Sales, color by TV_level
colors = {'Low': 'blue', 'High': 'red'}
for level, grp in df_adv.groupby('TV_level'):
    axes[1].scatter(grp['TV'], grp['Sales'], alpha=0.4,
                    label=f'{level} TV', color=colors[level])
axes[1].set_xlabel('TV'); axes[1].set_ylabel('Sales')
axes[1].set_title('TV vs Sales by Level'); axes[1].legend()

plt.tight_layout(); plt.show()

### TODO 1 (Easy): ตีความ Dummy Coefficient และ Add Continuous Predictor

**สิ่งที่ต้องทำ**:
1. จาก summary: β̂_TV_level[T.High] คืออะไร? Sales ของ High-TV สูงกว่า Low-TV เฉลี่ยเท่าไร?
2. Fit model ใหม่: `Sales ~ TV + C(TV_level)` — TV (continuous) + TV_level (categorical) พร้อมกัน
3. อธิบาย: ทำไมถึงไม่ควรใส่ TV และ TV_level (ที่สร้างจาก TV) ใน model เดียวกัน?


In [ ]:
# TODO 1: ตีความ dummy + fit mixed model
# เติม code ที่นี่

---
## Part 2: Interaction Terms และ Polynomial Regression

**Part นี้เราจะเพิ่ม interaction term** TV:Radio เพื่อจับ synergy effect และ polynomial feature สำหรับ non-linear relationship

Interaction term จำเป็นเมื่อ effect ของ X₁ ต่อ Y ขึ้นกับค่าของ X₂ ตัวอย่างคลาสสิกคือ TV และ Radio ในการโฆษณา — ถ้าใช้ทั้งสองพร้อมกัน ผลรวมอาจมากกว่าที่ additive model คาดไว้


In [ ]:
# ─── Additive vs Interaction model ─────────────────────────────────
# วัตถุประสงค์: แสดงว่า interaction term เพิ่ม R² จาก 0.90 → 0.97
m_add = smf.ols('Sales ~ TV + Radio', data=df_adv).fit()
m_int = smf.ols('Sales ~ TV + Radio + TV:Radio', data=df_adv).fit()

print('Additive model:')
print(f'  R² = {m_add.rsquared:.4f}, RSE = {np.sqrt(m_add.mse_resid):.4f}')
print('\nInteraction model:')
print(f'  R² = {m_int.rsquared:.4f}, RSE = {np.sqrt(m_int.mse_resid):.4f}')
print('\nInteraction model coefficients:')
print(m_int.params.round(6))

### TODO 2 (Medium): Polynomial Regression บน Auto Data

**สิ่งที่ต้องทำ**:
1. Fit 3 models: linear, quadratic, degree-5 polynomial บน Auto data (mpg ~ horsepower)
2. เปรียบเทียบ R² และ RSE
3. Plot scatter + ทั้ง 3 fitted curves ในรูปเดียวกัน
4. ตีความ: polynomial degree ใดเหมาะสม? ทำไม degree-5 ไม่ดีกว่า degree-2 มาก?


In [ ]:
# TODO 2: polynomial regression
# วัตถุประสงค์: เปรียบเทียบ linear vs quadratic vs high-degree polynomial
# เติม code ที่นี่
# Hint: ใช้ I(horsepower**2) ใน formula หรือ PolynomialFeatures จาก sklearn

---
## Part 3: Residual Diagnostics — 4 Standard Plots

**Part นี้เราจะทำ diagnostic plots** ที่ Data Scientist ต้องทำทุกครั้งหลัง fit regression model เพื่อตรวจสอบว่า assumptions ทั้ง 4 ข้อ (LINE) เป็นจริง

เราจะใช้ linear model (mpg ~ horsepower) เพื่อแสดง violation ชัดๆ แล้วเปรียบเทียบกับ quadratic model ที่ fix แล้ว


In [ ]:
# ─── Fit linear model สำหรับ diagnostics ──────────────────────────
# วัตถุประสงค์: ดู violations จาก linear model ก่อน แล้วค่อย fix
m_linear = smf.ols('mpg ~ horsepower', data=df_auto).fit()
m_quad   = smf.ols('mpg ~ horsepower + I(horsepower**2)', data=df_auto).fit()

def plot_diagnostics(model, title='Regression Diagnostics'):
    """สร้าง 4 diagnostic plots สำหรับ regression model"""
    # วัตถุประสงค์: standard 4-panel diagnostic plot เหมือน R's plot(lm())
    influence = OLSInfluence(model)
    fitted    = model.fittedvalues
    resid     = model.resid
    stud_resid = influence.resid_studentized_external
    leverage   = influence.hat_matrix_diag
    cooks_d    = influence.cooks_distance[0]

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    fig.suptitle(title, fontsize=14)

    # 1. Residuals vs Fitted
    axes[0,0].scatter(fitted, resid, alpha=0.5, s=20)
    axes[0,0].axhline(0, color='red', lw=1.5)
    axes[0,0].set_xlabel('Fitted values')
    axes[0,0].set_ylabel('Residuals')
    axes[0,0].set_title('Residuals vs Fitted')

    # 2. QQ-plot
    ProbPlot(resid).qqplot(line='s', ax=axes[0,1])
    axes[0,1].set_title('Normal Q-Q')

    # 3. Scale-Location
    sqrt_abs = np.sqrt(np.abs(stud_resid))
    axes[1,0].scatter(fitted, sqrt_abs, alpha=0.5, s=20)
    axes[1,0].set_xlabel('Fitted values')
    axes[1,0].set_ylabel('√|Studentized Residuals|')
    axes[1,0].set_title('Scale-Location')

    # 4. Leverage vs Cook's Distance
    axes[1,1].scatter(leverage, cooks_d, alpha=0.5, s=20)
    axes[1,1].axhline(4/len(fitted), color='r', lw=1.5, linestyle='--',
                      label=f'Cook\'s = 4/n={4/len(fitted):.4f}')
    avg_lev = (model.df_model + 1) / len(fitted)
    axes[1,1].axvline(2*avg_lev, color='orange', lw=1.5, linestyle='--',
                      label=f'Leverage = 2×avg={2*avg_lev:.4f}')
    axes[1,1].set_xlabel('Leverage')
    axes[1,1].set_ylabel("Cook's Distance")
    axes[1,1].set_title("Leverage vs Cook's Distance")
    axes[1,1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()
    return fig

# Diagnostic plots: linear model
plot_diagnostics(m_linear, 'Linear: mpg ~ horsepower (BEFORE fix)')

### TODO 3 (Medium): อ่าน Diagnostic Plots และ Fix

**สิ่งที่ต้องทำ**:
1. จาก 4 diagnostic plots ของ linear model — ระบุปัญหาที่เห็น:
   - Residuals vs Fitted: มี pattern อะไร?
   - QQ-plot: normality OK ไหม?
   - Scale-Location: homoscedasticity OK ไหม?
   - Leverage: มี influential points ไหม?
2. Plot diagnostics ของ quadratic model (m_quad) เปรียบเทียบ — ดีขึ้นไหม?
3. ระบุ observations ที่มี studentized residual |r| > 2 จาก linear model


In [ ]:
# TODO 3: อ่านผล diagnostics + fix
# วัตถุประสงค์: เปรียบเทียบ diagnostic plots ก่อนและหลัง fix polynomial

# เติม code ที่นี่
# 1. plot_diagnostics(m_quad, 'Quadratic: mpg ~ hp + hp² (AFTER fix)')
# 2. หา outliers: np.where(np.abs(stud_resid) > 2)[0]
# 3. เขียน markdown cell อธิบายปัญหาที่เห็น

---
## Part 4: VIF — Multicollinearity Detection

**Part นี้เราจะตรวจ multicollinearity** โดยใช้ Variance Inflation Factor (VIF) ซึ่งวัดว่า SE ของ β̂ พองขึ้นจากการที่ predictors correlated กันเท่าไร

เราจะใช้ Advertising dataset (TV + Radio + Newspaper) ที่มี collinearity ระดับปานกลาง เปรียบเทียบกับ synthetic dataset ที่มี collinearity สูงมาก


In [ ]:
# ─── คำนวณ VIF สำหรับ Advertising dataset ──────────────────────────
# วัตถุประสงค์: ตรวจ multicollinearity ใน TV+Radio+Newspaper model
X_adv = sm.add_constant(df_adv[['TV', 'Radio', 'Newspaper']])
vif_adv = pd.DataFrame({
    'Feature': X_adv.columns,
    'VIF': [variance_inflation_factor(X_adv.values, i)
            for i in range(X_adv.shape[1])]
})
print('VIF — Advertising dataset:')
print(vif_adv.round(2))
print('\nAll VIF < 5 → Collinearity OK ✓')

In [ ]:
# ─── High collinearity example ──────────────────────────────────────
# วัตถุประสงค์: แสดงให้เห็นว่า VIF >> 10 เกิดขึ้นอย่างไรและมีผลอย่างไร
np.random.seed(42)
n = 200
X1 = np.random.normal(0, 1, n)
X2 = 0.99 * X1 + 0.01 * np.random.normal(0, 1, n)  # correlated มาก
X3 = np.random.normal(0, 1, n)                       # independent
y  = 2*X1 + 3*X2 + 1.5*X3 + np.random.normal(0, 1, n)
df_collinear = pd.DataFrame({'X1': X1, 'X2': X2, 'X3': X3, 'y': y})

X_col = sm.add_constant(df_collinear[['X1','X2','X3']])
vif_col = pd.DataFrame({
    'Feature': X_col.columns,
    'VIF': [variance_inflation_factor(X_col.values, i)
            for i in range(X_col.shape[1])]
})
print('VIF — High collinearity (X1≈X2):')
print(vif_col.round(2))

m_collinear = smf.ols('y ~ X1 + X2 + X3', data=df_collinear).fit()
print('\nt-statistics:')
print(m_collinear.tvalues.round(3))
print('\nNote: X1 and X2 both have large true effects (2 and 3) but small t!')

### TODO 4 (Hard): Fix Multicollinearity + Full VIF Analysis

**สิ่งที่ต้องทำ**:
1. สำหรับ collinear dataset: ลบ X2 ออก แล้ว fit `y ~ X1 + X3`
   - เปรียบ SE(β̂_X1) ก่อน-หลัง
   - เปรียบ t-statistic ของ X1 ก่อน-หลัง
2. สำหรับ Advertising: ทำ correlation heatmap + VIF + บอกว่า predictor ใดมีปัญหาหรือไม่
3. สร้าง visualization: bar chart เปรียบ VIF ของทุก predictor พร้อมเส้น threshold ที่ VIF=5


In [ ]:
# TODO 4: Fix multicollinearity
# วัตถุประสงค์: แสดงว่าการลบ predictor ที่ collinear ช่วยให้ SE และ t-stat ดีขึ้น

# เติม code ที่นี่

---
## Part 5: Case Study — Housing Price Diagnostics

**Scenario**: บริษัทอสังหาริมทรัพย์ต้องการ model ราคาบ้าน (medv) จากคุณลักษณะของบ้าน และทีม Data Science ต้องตรวจสอบ assumptions ก่อนเสนอ model ให้ผู้บริหาร

ใช้ Boston Housing dataset (synthetic version)


In [ ]:
# ─── Boston Housing (synthetic) ────────────────────────────────────
# วัตถุประสงค์: dataset ที่มีลักษณะเหมือน Boston Housing สำหรับ full diagnostics
np.random.seed(99)
n = 506
lstat  = np.random.uniform(1.7, 38, n)   # % lower status population
rm     = np.random.uniform(3.6, 8.8, n)  # average rooms per dwelling
tax    = np.random.uniform(187, 711, n)  # property tax rate
ptratio= np.random.uniform(12, 22, n)    # pupil-teacher ratio
# true relationship with some non-linearity
medv = (50 - 0.6*lstat + 5*rm - 0.01*tax - 0.3*ptratio
        - 0.01*lstat**2 + np.random.normal(0, 5, n))
medv = np.clip(medv, 5, 50)  # ราคาระหว่าง 5–50 พัน$
df_boston = pd.DataFrame({'medv':medv,'lstat':lstat,'rm':rm,'tax':tax,'ptratio':ptratio})
print('Boston Housing (synthetic):', df_boston.shape)
print(df_boston.describe().round(2))

### TODO 5 (Hard): Full Diagnostics Workflow

**สิ่งที่ต้องทำ**:
1. Fit: `medv ~ lstat + rm + tax + ptratio` ด้วย statsmodels
2. สร้าง 4 diagnostic plots
3. คำนวณ VIF ของทุก predictor
4. ระบุปัญหาที่พบ (non-linearity, outliers, etc.)
5. Fix อย่างน้อย 1 ปัญหา (เช่น เพิ่ม `I(lstat**2)` ถ้า non-linearity)
6. เปรียบ model ก่อน-หลัง fix: R²_adj, AIC, diagnostic plots
7. เขียน markdown report 5–7 ประโยคสำหรับ CEO


In [ ]:
# TODO 5: Full diagnostics workflow
# วัตถุประสงค์: ฝึก end-to-end diagnostics ที่ต้องทำในงานจริง

# เติม code ที่นี่

---
## สรุป Lab 10

| Concept | Method | Python |
|---------|--------|--------|
| Dummy variable | C() ใน formula | `smf.ols('y ~ C(x)', data)` |
| Interaction | X1:X2 หรือ X1*X2 | `'y ~ x1 + x2 + x1:x2'` |
| Polynomial | I(X**2) | `'y ~ x + I(x**2)'` |
| Residual plot | scatter(fitted, resid) | `model.resid`, `model.fittedvalues` |
| QQ-plot | probplot | `ProbPlot(resid).qqplot(line='s')` |
| Leverage | hat matrix diagonal | `OLSInfluence(model).hat_matrix_diag` |
| Cook's D | influence measure | `OLSInfluence(model).cooks_distance[0]` |
| VIF | 1/(1−R²_{Xⱼ}) | `variance_inflation_factor(X, j)` |

## Reflection Questions

1. **Hierarchy principle**: ทำไมถ้าใส่ interaction term TV:Radio ต้องใส่ TV และ Radio ด้วยแม้ p-value ของ main effects จะสูง?

2. **Diagnostic order**: ถ้าต้องตรวจ assumptions ทั้ง 4 ข้อ (LINE) คุณจะเรียงลำดับการตรวจอย่างไร และปัญหาข้อใดร้ายแรงที่สุดถ้าละเมิด?

3. **VIF vs correlation**: ทำไม VIF ดีกว่า correlation matrix ในการตรวจ multicollinearity? กรณีใดที่ correlation matrix พลาดแต่ VIF จับได้?
